In [1]:
import os
from pathlib import Path
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm
import shutil

PROJECT_ROOT = Path(r"E:\Python Projects\Celebrity_Face_matcher")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
METADATA_DIR = PROJECT_ROOT / "data" / "metadata"

# Load the registry from Notebook 1
registry_path = METADATA_DIR / "celebrity_registry.csv"
if not registry_path.exists():
    raise FileNotFoundError("Please run Notebook 01 first to generate the registry.")

df_registry = pd.read_csv(registry_path)
print(f"Loaded {len(df_registry)} celebrity classes.")

# Target dimensions for VGGFace ResNet50
TARGET_SIZE = (224, 224)
# Max images per class to prevent Hollywood from dominating Bollywood
MAX_IMAGES_PER_CLASS = 80

Loaded 205 celebrity classes.


In [2]:
VALID_EXTS = {".jpg", ".jpeg", ".png", ".webp"}
processed_records = []

for _, row in tqdm(df_registry.iterrows(), total=len(df_registry), desc="Processing Identities"):
    source_dir = Path(row['raw_staging_path'])
    target_dir = PROCESSED_DIR / row['folder_slug']
    
    # Create the clean class directory
    target_dir.mkdir(parents=True, exist_ok=True)
    
    valid_count = 0
    
    # Grab all valid image files in the source directory
    img_files = [f for f in source_dir.iterdir() if f.is_file() and f.suffix.lower() in VALID_EXTS]
    
    for img_path in img_files:
        if valid_count >= MAX_IMAGES_PER_CLASS:
            break # Cap the dataset to maintain class balance
            
        try:
            # 1. Open and verify image
            with Image.open(img_path) as img:
                img.verify() # Checks for file corruption
            
            # 2. Re-open to manipulate (verify() closes the file)
            with Image.open(img_path) as img:
                # Force RGB (removes alpha channels from PNGs/WebPs)
                if img.mode != 'RGB':
                    img = img.convert('RGB')
                
                # Resize to VGGFace expected input size
                img_resized = img.resize(TARGET_SIZE, Image.Resampling.LANCZOS)
                
                # Standardize name: e.g., aamir_khan_001.jpg
                new_filename = f"{row['folder_slug']}_{valid_count+1:03d}.jpg"
                save_path = target_dir / new_filename
                
                # Save as high-quality JPEG
                img_resized.save(save_path, "JPEG", quality=95)
                valid_count += 1
                
        except Exception as e:
            # Skip corrupted or unreadable images silently
            continue
            
    processed_records.append({
        "celebrity_id": row["celebrity_id"],
        "folder_slug": row["folder_slug"],
        "processed_image_count": valid_count
    })

# Update registry with actual processed counts
df_processed = pd.DataFrame(processed_records)
df_final = pd.merge(df_registry, df_processed, on=["celebrity_id", "folder_slug"])
df_final.to_csv(METADATA_DIR / "celebrity_registry_processed.csv", index=False)

print("\nProcessing complete! Unified dataset is ready in data/processed/")

Processing Identities:   0%|          | 0/205 [00:00<?, ?it/s]


Processing complete! Unified dataset is ready in data/processed/
